# BARseq gene-panel design — analysis

Consolidated analysis of the selected panels. All logic lives in the importable modules
(loaded as top-level modules to bypass the package `__init__`, which pulls in
`iss_preprocess`):

- **`panel_design.py`** — selection pipeline (streaming loader, filter, MI, greedy, fusion)
- **`panel_eval.py`** — standardised dataset loaders, `make_bundle`, `evaluate_panel`, reference panels
- **`panel_plots.py`** — all plotting / evaluation helpers used below

Selection runs are in `results/panel*` (CLI: `sbatch panel_design.sh`). Set `PANEL_DIR`
below to choose which run to inspect (`results/panel` = min_expr 3, `results/panel_minexpr1`
= min_expr 1).

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO / 'iss_analysis'))   # bypass package __init__
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import panel_eval as pe, panel_plots as pp

PANEL_DIR = str(REPO / 'results' / 'panel')        # or 'results/panel_minexpr1'
ds2020 = pe.load_allen2020()
combined, ranking = pp.load_ranking(PANEL_DIR)
panels = pp.standard_panels(PANEL_DIR, ds2020)     # classification / mapping / combined
refs = pe.reference_panels(set(ds2020['gene_names']))
print('combined panel:', len(combined), '| panels:', {k: len(v) for k, v in panels.items()})
ranking.head(15)

## 1. Selection curves (accuracy + manifold preservation vs panel size)

In [ ]:
pp.selection_curves(PANEL_DIR); plt.show()

## 2. Expression across subclasses (ranked; red = peak subclass)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 13))
pp.expression_across_subclasses(ds2020, combined, ax=ax, max_genes=60)
plt.show()

## 3. Per-gene single-cell expression across subclasses
Writes a multi-page PDF (blue=excitatory, orange=inhibitory, red=mean). Use `combined`
for all 400; here the top 60 for speed.

In [ ]:
pp.per_gene_boxplots(ds2020, combined[:60], str(Path(PANEL_DIR) / 'per_gene_boxplots.pdf'))
print('wrote', Path(PANEL_DIR) / 'per_gene_boxplots.pdf')

## 4. VISp confusion matrices (subclass + cluster) by panel size

In [ ]:
pp.confusion_grid(ds2020, combined, level='subclass', region='VISp'); plt.show()
pp.confusion_grid(ds2020, combined, level='cluster', region='VISp'); plt.show()

## 5. UMAP grid — classification / mapping / combined + reference panels (by subclass)

In [ ]:
ref_rows = {f"codebook_88 ({len(refs['codebook_88'])}g)": refs['codebook_88'],
            f"csm_and_v1 ({len(refs['csm_and_v1'])}g)": refs['csm_and_v1']}
pp.umap_grid(ds2020, panels, refs=ref_rows, title='Allen 2020 (eff=0.1)'); plt.show()

## 6. Leiden clustering vs Allen subclass (combined top-400)

In [ ]:
fig, ari = pp.leiden_vs_labels(ds2020, combined[:400]); print('ARI', round(ari, 3)); plt.show()

## 7. Cross-panel / cross-dataset comparison
Our panel vs published Chen-lab panels on Allen 2020 and (independent) Allen 2018.

In [ ]:
ds2018 = pe.load_allen2018()                         # requires results/panel_cache_2018
combined18, _ = pp.load_ranking(str(REPO / 'results' / 'panel_2018'))
allp = {'panel_2020': combined, 'panel_2018': combined18, **refs}
tab = pp.panel_table({'data_2018': ds2018, 'data_2020': ds2020}, allp, sizes=(200, 400))
tab.pivot_table(index=['panel'], columns=['data', 'size'], values=['subclass', 'cluster'])

## 8. VISp vs MOs (within Allen 2020)

In [ ]:
acc = pp.region_accuracy(ds2020, combined, {'VISp': 'VISp', 'MOs': 'MOs_FRP'})
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for r in ['VISp', 'MOs']:
    d = acc[acc.region == r]
    ax[0].plot(d['size'], d['subclass'], '-o', label=r)
    ax[1].plot(d['size'], d['cluster'], '-o', label=r)
ax[0].set_title('subclass'); ax[1].set_title('cluster')
for a in ax: a.legend(); a.grid(alpha=.3); a.set_xlabel('# genes'); a.set_ylabel('accuracy')
plt.show()